# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [8]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
HF_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [ ]:
# Listing all available models from GROQ
import requests
import json

url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(json.dumps(response.json(), indent=2))

### Example 1

#### Create our QandA application

In [4]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser, BaseOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langchain.evaluation.qa import QAGenerateChain

In [6]:
file = r'./data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding="utf-8")
data = loader.load()

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(data)

# Create embeddings and vectorstore
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu', 'token': HF_TOKEN}
)

vectorstore = InMemoryVectorStore.from_documents(
    docs, embeddings
)

retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1054.51it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain.chains.combine_documents import create_stuff_documents_chain

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

<<<<>>>>>

QUESTION: {question}

ANSWER:
""")

qa = create_stuff_documents_chain(llm, prompt, document_separator="<<<<>>>>>")

# Fixed chain - provides both context AND question
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa
)


result = chain.invoke("Do the Cozy Comfort Pullover Set have side pockets?")
print(result)

Yes, the Cozy Comfort Pullover Set has side pockets. According to the description, the "Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg."


#### Coming up with test datapoints

In [11]:
data[10]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [12]:
data[11]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)


class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)


llm = ChatGroq(model="llama-3.3-70b-versatile")
llm_chain = prompt_template | llm | AnswerOutputParser()

query = "Is the Cozy Comfort Pullover Set available in different colors?"
result = llm_chain.invoke({"query": query})
print(result.answer)

Yes


In [14]:
# Run this to see what's in langchain.evaluation
import langchain.evaluation.qa
print(dir(langchain.evaluation.qa))

['ContextQAEvalChain', 'CotQAEvalChain', 'QAEvalChain', 'QAGenerateChain', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'eval_chain', 'eval_prompt', 'generate_chain', 'generate_prompt']


#### LLM-Generated examples

In [16]:
from langchain.evaluation.qa import QAGenerateChain

example_gen_chain = QAGenerateChain.from_llm(ChatGroq(model="llama-3.3-70b-versatile"))

new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

d_flattened = [data['qa_pairs'] for data in new_examples]

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\3425191476.py:5: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(


#### Combine examples

In [17]:
# examples += new_example
examples += d_flattened

In [18]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What type of material is used for the innersole of the Women's Campside Oxfords, and what feature does it have for odor control?",
  'answer': "The Women's Campside Oxfords have a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control."},
 {'query': 'What are the dimensions of the Small and Medium Recycled Waterhog Dog Mats, and what is the percentage of recycled materials used in the construction of the mat?',
  'answer': 'The dimensions of the Small Recycled Waterhog Dog Mat are 18" x 28", and the dimensions of the Medium Recycled Waterhog Dog Mat are 22.5" x 34.5". The mat is constructed from 24 oz. polyester fabric made from 94% recycled materials.'},
 {'query': "What percentage of the sun's harmful rays does the UPF 50+ rated fabric in the Inf

In [19]:
result = qa.invoke({
    "question": examples[6]["query"],  # ← "question" not "input"
    "context": retriever.invoke(examples[6]["query"])
})
print(result)

The primary material used in the EcoFlex 3L Storm Pants is 100% nylon, exclusive of trim. The recommended care instructions for the pants are to machine wash and dry.


### Manual Evaluation - Fun part

In [20]:
import langchain
langchain.debug = True

# qa needs BOTH question AND context
result = qa.invoke({
    "question": examples[0]["query"],    
    "context": retriever.invoke(examples[0]["query"])  
})
print(result)

[chain/start] [chain:stuff_documents_chain] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context>] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] Entering Chain run with input:
[inputs]
[chain/end] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] s] Exiting Chain run with output:
{
  "output": "On-seam pockets, with a zippered security pocket on the right side.\n\nAll pockets have sturdy pocket bags and offer plenty of room for a wallet, cell phone and more.\n\nGusseted crotch for ease of movement.\n\nImported.<<<<>>>>>: 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name.

In [21]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [22]:
# predictions must be list of dicts like:
# [{'query': '...', 'answer': '...', 'result': '...'}, ...]

predictions = []
for i, eg in enumerate(examples[:5]):  # Limit for testing
    docs = retriever.invoke(eg["query"])
    result = qa.invoke({"question": eg["query"], "context": docs})
    
    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result
    }
    predictions.append(pred_dict)

# Now your print loop works perfectly!
for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set has side pockets. According to the description, the "Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg."

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 2:
Question: What type of material is used for the innersole of the Women's Campside Oxfords, and what feature does it have for odor control?
Real Answer: The Women's Campside Oxfords have a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control.
Predicted Answer: The material used for the innersole of the Women's Campside Oxfords is a comfortable EVA (Ethylene-Vinyl Acetate) innersole, and it features Cleansport NXT antimicrobial odor control.

Ex

In [24]:
import re
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score how well PREDICTED matches ANSWER for the given QUERY.
Respond with ONLY a single decimal number between 0 and 1. No explanation, no text, just the number.

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()

graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    # Regex fallback in case the model adds extra text
    match = re.search(r'\b([01](?:\.\d+)?)\b', score_raw)
    score = float(match.group(1)) if match else 0.0

    graded_outputs.append({'score': score})

    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real: Yes
Predicted: Yes, the Cozy Comfort Pullover Set has side pockets. According to the description, the "Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg."
Score: 1.0 

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real: The DownTek collection
Predicted: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Score: 0.9 

Example 2:
Question: What type of material is used for the innersole of the Women's Campside Oxfords, and what feature does it have for odor control?
Real: The Women's Campside Oxfords have a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control.
Predicted: The material used for the innersole of the Women's Campside Oxfords is a comfortable EVA (Ethylene-Vinyl Acetate) innersole, and it features Cleansport NXT antimicrobial odor control.
Score: 0.9 

Example 

In [25]:
graded_outputs

[{'score': 1.0},
 {'score': 0.9},
 {'score': 0.9},
 {'score': 0.9},
 {'score': 0.9}]

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [27]:
from langchain_community.document_loaders import TextLoader
from langchain_core.runnables import RunnablePassthrough

# Load the text file
loader = TextLoader(r"./data/nyc_text.txt", encoding="utf-8")
docs = loader.load()

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)

# Create vectorstore
vectorstore = InMemoryVectorStore.from_documents(
    split_docs, HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu', 'token': HF_TOKEN}
    )
)
retriever = vectorstore.as_retriever()

# Create QA chain
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

QUESTION: {question}

ANSWER:
""")

qa_chain = create_stuff_documents_chain(llm, prompt)

# Combine into runnable chain that returns both answer and source documents
qa_chain_with_docs = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa_chain
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1064.98it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain_with_docs.invoke(question)
print(result)

New York City was named after King Charles II of England granted the lands to his brother, the Duke of York, in 1664, when the city came under British control. It was previously known as New Amsterdam, which was the name given to it by the Dutch colonists in 1626. The city was briefly renamed New Orange in 1673, but has been continuously known as New York since November 1674.


Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [29]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [30]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [31]:
# Configure ragas to use Groq (instead of OpenAI) and local HuggingFace embeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.3-70b-versatile", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu', 'token': HF_TOKEN}
    )
)

print("Ragas configured with Groq LLM and HuggingFace embeddings ✓")

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\4192442755.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.3-70b-versatile", temperature=0))
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1038.57it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ragas configured with Groq LLM and HuggingFace embeddings ✓


C:\Users\Work\AppData\Local\Temp\ipykernel_49896\4192442755.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [32]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

all_predictions = []
for example in examples:
    q = example["query"]
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})
    all_predictions.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": example["ground_truths"][0]
    })

dataset_all = Dataset.from_list(all_predictions)

print("evaluating faithfulness...")
f = evaluate(dataset_all, metrics=[faithfulness], llm=ragas_llm, embeddings=ragas_embeddings)
print(f)

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1373752801.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1373752801.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


evaluating faithfulness...


Evaluating: 100%|██████████| 5/5 [00:14<00:00,  2.93s/it]


{'faithfulness': 0.8810}


1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

**Faithfulness**

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [33]:
print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall], llm=ragas_llm, embeddings=ragas_embeddings)
print(r)

evaluating context recall...


Evaluating: 100%|██████████| 5/5 [00:29<00:00,  5.82s/it]


{'context_recall': 0.6333}


In [35]:
from ragas import evaluate
from ragas.metrics import faithfulness
from datasets import Dataset

def get_score(result, metric_name):
    """Extract a float from ragas result regardless of whether it returns a scalar or list."""
    score = result[metric_name]
    return float(score[0]) if isinstance(score, list) else float(score)

# Use question 2 (economic significance) — produces long detailed answer
high_data = Dataset.from_list([all_predictions[2]])
high_score = get_score(evaluate(high_data, [faithfulness], llm=ragas_llm, embeddings=ragas_embeddings), "faithfulness")
print("HIGH Faithfulness (real answer):", high_score)

# LOW — contradicting claims
fake = all_predictions[2].copy()
fake["answer"] = "New York City has no economic importance. It is a small rural town with no financial institutions, no corporations, and no international trade. Wall Street closed in 1990."
low_data = Dataset.from_list([fake])
low_score = get_score(evaluate(low_data, [faithfulness], llm=ragas_llm, embeddings=ragas_embeddings), "faithfulness")
print("LOW Faithfulness (fake answer):", low_score)

print("\n--- Explanation ---")
print(f"Real answer score:  {high_score:.2f}  ← claims supported by context")
print(f"Fake answer score:  {low_score:.2f}  ← claims contradict context")

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\3680468164.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
Evaluating: 100%|██████████| 1/1 [00:05<00:00,  5.28s/it]


HIGH Faithfulness (real answer): 1.0


Evaluating: 100%|██████████| 1/1 [00:02<00:00,  2.42s/it]


LOW Faithfulness (fake answer): 0.0

--- Explanation ---
Real answer score:  1.00  ← claims supported by context
Fake answer score:  0.00  ← claims contradict context


In [36]:
print("Question:", all_predictions[2]["question"])
print("Answer:", all_predictions[2]["answer"])

Question: What is the economic significance of New York City?
Answer: New York City is a global hub of business and commerce, and is sometimes described as the capital of the world. It is a center for various industries such as banking and finance, healthcare, technology, retailing, trade, tourism, real estate, media, advertising, and the arts. The city has the largest metropolitan economy in the world, with a gross metropolitan product of over $2.4 trillion, and is home to the highest number of billionaires, ultra-high net worth individuals, and millionaires. If the New York metropolitan area were a sovereign state, it would have the eighth-largest economy in the world, making it an established safe haven for global investors.


**Context Relevancy**

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [37]:
from ragas.metrics import context_recall

# HIGH — real docs
high_data = Dataset.from_list([all_predictions[2]])
high_score = get_score(evaluate(high_data, [context_recall], llm=ragas_llm, embeddings=ragas_embeddings), "context_recall")
print("HIGH Context Recall (real docs):", high_score)

# LOW — off-topic docs
fake = all_predictions[2].copy()
fake["contexts"] = [
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language used in data science."
]
low_data = Dataset.from_list([fake])
low_score = get_score(evaluate(low_data, [context_recall], llm=ragas_llm, embeddings=ragas_embeddings), "context_recall")
print("LOW Context Recall (fake docs):", low_score)

print("\n--- Explanation ---")
print(f"Real docs score:  {high_score:.2f}  ← ground truth found in context")
print(f"Fake docs score:  {low_score:.2f}  ← ground truth absent from context")

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\522485355.py:1: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall
Evaluating: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]


HIGH Context Recall (real docs): 1.0


Evaluating: 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]


LOW Context Recall (fake docs): 0.0

--- Explanation ---
Real docs score:  1.00  ← ground truth found in context
Fake docs score:  0.00  ← ground truth absent from context


In [39]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

def get_contexts_str(contexts):
    return [
        doc.page_content if hasattr(doc, "page_content") else str(doc)
        for doc in contexts
    ] if contexts and hasattr(contexts[0], "page_content") else contexts

def evaluate_answer(question, answer, contexts, ground_truth=None, test_name=""):
    contexts_str = get_contexts_str(contexts)

    data = {
        "question": [question],
        "answer":   [answer],
        "contexts": [contexts_str],
        "ground_truth": [ground_truth if ground_truth else answer]
    }

    dataset = Dataset.from_dict(data)
    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings
    )

    print(f"\n{'='*60}")
    print(f"TEST: {test_name}")
    print(f"Faithfulness:     {get_score(results, 'faithfulness'):.4f}")
    print(f"Answer Relevancy: {get_score(results, 'answer_relevancy'):.4f}")
    print(f"Context Recall:   {get_score(results, 'context_recall'):.4f}")
    return results

# Test it
evaluate_answer(
    question=all_predictions[2]["question"],
    answer=all_predictions[2]["answer"],
    contexts=all_predictions[2]["contexts"],
    ground_truth=all_predictions[2]["ground_truth"],
    test_name="Economic Significance"
)

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1452160272.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1452160272.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1452160272.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_re


TEST: Economic Significance
Faithfulness:     1.0000
Answer Relevancy: 0.7765
Context Recall:   1.0000


{'faithfulness': 1.0000, 'answer_relevancy': 0.7765, 'context_recall': 1.0000}

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [40]:
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

dataset_all = Dataset.from_list(all_predictions)

results = evaluate(
    dataset_all,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

print(results)
results.to_pandas()

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1298980884.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1298980884.py:1: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\Work\AppData\Local\Temp\ipykernel_49896\1298980884.py:1: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

{'faithfulness': nan, 'answer_relevancy': 0.6488, 'context_precision': 0.8500, 'context_recall': 0.7500}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the population of New York City as of ...,[New York City is the most populous city in th...,"8,804,190","8,804,190",NaN,0.565753,1.00,1.0
1,Which borough of New York City has the highest...,[New York City is the most populous city in th...,The context does not explicitly state which bo...,Brooklyn,NaN,0.000000,0.75,NaN
2,What is the economic significance of New York ...,[Despite New York's heavy reliance on its vast...,New York City is a global hub of business and ...,"New York City's economic significance is vast,...",NaN,0.776520,0.50,NaN
3,How did New York City get its name?,[The city and its metropolitan area constitute...,New York City was named after King Charles II ...,New York City got its name when it came under ...,NaN,1.000000,1.00,0.5
4,What is the significance of the Statue of Libe...,[the Dutch in July 1673 and was renamed New Or...,The Statue of Liberty is a symbol of the U.S. ...,The Statue of Liberty in New York City holds g...,NaN,0.901801,1.00,NaN


In [41]:
# run the queries as a batch for efficiency
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

# Step 1 — batch predictions (same idea as qa_chain.batch(examples))
all_preds = []
for example in examples:
    q = example["query"]
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})
    all_preds.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": example["ground_truths"][0]
    })

dataset_all = Dataset.from_list(all_preds)

# Step 2 — replaces faithfulness_chain.evaluate(examples, predictions)
print("evaluating faithfulness...")
r = evaluate(dataset_all, metrics=[faithfulness], llm=ragas_llm, embeddings=ragas_embeddings)
print(r)

# Step 3 — replaces context_recall_chain.evaluate(examples, predictions)
print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall], llm=ragas_llm, embeddings=ragas_embeddings)
print(r)

C:\Users\Work\AppData\Local\Temp\ipykernel_49896\803257790.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
C:\Users\Work\AppData\Local\Temp\ipykernel_49896\803257790.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kj5atb4nese86avfgrjhwe4v` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 688. Please try again in 9m54.432s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Evaluation Results — Analysis

---

### Example 1 — Outdoor Clothing Catalog (Manual LLM Evaluation)

**Scores: 0.9–1.0 across 5 examples**

The LLM judge scored factual accuracy well across all examples. The only example that reached 1.0 was the simple yes/no question (Example 0), while the rest scored 0.9 — not because the answers were wrong, but because the predictions were more verbose than the reference answers.

For instance:
- **Reference:** `"The DownTek collection"`
- **Predicted:** `"The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection."`

The judge correctly treats these as semantically equivalent but not a perfect match. This illustrates why scoring-based evaluation is preferable to exact string matching — a verbatim match would have failed here despite the answer being entirely correct.

---

### Example 2 — NYC Text (RAGAS Evaluation)

#### Overall batch results

| Metric | Score | Interpretation |
|---|---|---|
| **Faithfulness** | 0.881 | Strong — model stays grounded in retrieved context |
| **Context Recall** | 0.633 | Weak — ground truth not always present in retrieved chunks |

**Faithfulness at 0.881** indicates the model is mostly generating answers supported by the retrieved context rather than hallucinating. The few points lost likely come from answers that blend retrieved facts with general LLM knowledge.

**Context Recall at 0.633** is the weakest result in the notebook. It measures whether the retrieved chunks actually contain the information needed to answer correctly. The retriever (`all-MiniLM-L6-v2` with default top-k) is not consistently surfacing the most relevant passages, especially for complex or multi-part questions.

#### Faithfulness and Context Recall — high/low demos

Both demos produced clean **1.0 vs 0.0** separations, which validates that the metrics are working as meaningful signals and not just noise:
- A real answer grounded in context → Faithfulness = 1.0
- A fabricated, contradictory answer → Faithfulness = 0.0
- Real retrieved documents → Context Recall = 1.0
- Off-topic documents (Eiffel Tower, Python language) → Context Recall = 0.0

---

#### Full 4-metric evaluation

| Metric | Score | Notes |
|---|---|---|
| **Faithfulness** | `NaN` | Groq's 100k token/day limit was reached mid-evaluation → RateLimitErrors and TimeoutErrors |
| **Answer Relevancy** | 0.649 | Moderate — dragged down significantly by Q1 |
| **Context Precision** | 0.850 | Good — retrieved chunks are mostly on-topic |
| **Context Recall** | 0.750 | Moderate — ground truth not always captured in retrieval |

#### Per-question breakdown

| Question | Answer Relevancy | Context Precision | Context Recall |
|---|---|---|---|
| Population (2020)? | 0.57 | 1.00 | 1.00 |
| Which borough has the highest population? | **0.00** | 0.75 | NaN |
| What is the economic significance of NYC? | 0.78 | 0.50 | NaN |
| How did NYC get its name? | **1.00** | 1.00 | 0.50 |
| What is the significance of the Statue of Liberty? | 0.90 | 1.00 | NaN |

**The most instructive case is Q1 (borough).** The model responded with `"The context does not explicitly state which borough..."` — which is factually wrong since Brooklyn is the answer and that information exists in the source document. The retriever failed to surface the relevant chunk, so the model hedged rather than answering. Answer relevancy correctly scored this **0.00** because a non-answer is irrelevant to the question. This is a clear example of how a context recall failure cascades into an answer quality failure.

**The NaN faithfulness scores** are a practical constraint of the free Groq tier (100k tokens/day). Running 5 questions × 4 metrics in parallel consumes the daily budget mid-evaluation. This is not a code issue — it is a rate limit that would be resolved with a paid API tier or by spacing evaluations across multiple sessions.

---

### Key Takeaways

1. **The RAG pipeline performs well on direct factual questions** (names, numbers, explicit descriptions) but struggles when the relevant information requires the retriever to find exactly the right chunk among many candidates.

2. **Context recall (0.63–0.75) is the bottleneck**, not the LLM. Improving retrieval — larger top-k, better embeddings, or a reranker — would likely lift both recall and final answer quality without changing the model at all.

3. **API rate limits are a real operational constraint.** Free-tier token budgets are insufficient for running comprehensive RAGAS evaluations in a single session. This is an important consideration when designing evaluation pipelines for production use.

4. **The high/low metric demos confirm the metrics are meaningful.** A 1.0 vs 0.0 split between grounded and fabricated answers gives confidence that faithfulness and context recall are measuring something real and actionable.